# IHP SG13G2 L2n0: WIP measured-data comparison

This notebook compares the exploratory gsim/Palace L2n0 model with both
measured two-port files and the published IHP OpenEMS results. The recorded
Palace model does not yet reproduce the measurements; its geometry mapping,
ports, materials and mesh limitations are documented below.

The primary observables are all four complex S-parameters and differential
impedance, inductance and quality factor. Two measured differential one-port
files are shown as a separate experiment. All four measured files and seven
published OpenEMS outputs are pinned with provenance and bundled locally.

The recorded simulations can be inspected without network access or launching
a solver. This example remains WIP until the model differences and numerical
convergence checks are resolved.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import skrf as rf

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "scripts" / "ihp_l2n0_benchmark.py").is_file():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("run this notebook from the gsim root or nbs directory")
from scripts import ihp_l2n0_benchmark as b

plt.rcParams["figure.dpi"] = 120
s_labels = ((0, 0, "S11"), (1, 0, "S21"), (0, 1, "S12"), (1, 1, "S22"))

## L2n0 — WIP exploratory Palace comparison

The recorded Palace sweep is compared with both pinned, THRU-de-embedded
measured two-port `.s2p` files. The complete complex S-matrix is the primary
observable, with 50 Ω reference impedance. The 11 Palace frequencies span
1–50 GHz; measured complex S-values are interpolated onto those points.

The Palace excitation uses interlayer ports and a minimal SUBGND pad, whereas
the pinned IHP two-port OpenEMS model uses TopMetal1 edge ports and a
Metal1/common-ground pillar. The present Palace model also has unresolved mesh
error and does not reproduce the measured two-port response. Its 1 GHz
`2 Z0 (1-S21)/S21` agreement assumes a pure series element and is only a
diagnostic, not validation evidence. The published OpenEMS result is displayed
separately below; no new OpenEMS or Palace solve was run for this notebook.


### Pinned GDS and recorded Palace mesh

The left panel renders the vendored `L_2n0_simplified.gds`; the right panel
shows TopMetal1/TopMetal2/port-sheet surface triangles from the actual 5 µm
`ground0_pec` Palace mesh (15,260 nodes, 87,482 tetrahedra). Its mesh metadata
matches the recorded run. GDS layer 75 is shown in gray but is **unmapped by
the current Palace stack**; the recorded model instead adds a SUBGND pad.
The mesh picture establishes geometry and discretization provenance, not
convergence. The local `.msh` is not committed; the [recorded metadata](data/ihp_validation/experiments/ground0_pec/metadata.json)
and the generated image are retained.

![L2n0 pinned GDS and recorded Palace surface mesh](../docs/assets/plots/ihp-l2n0-gds-and-palace-mesh.png)


In [ ]:
RUN = b.RUN_ROOT / "l2n0_best"
run = b.load_recorded_run(RUN)
sp = run.sparams
print("Palace sweep:", sp.port_names, np.asarray(sp.freq), "GHz")

In [ ]:
# All four complex S-parameters, with measured data sampled at Palace frequencies.
# Both measured files and Palace use a 50-ohm reference impedance.
from matplotlib.ticker import MaxNLocator

f_sim = np.asarray(sp.freq, dtype=float)
s_sim = np.stack([b.s_matrix(sp, i) for i in range(len(f_sim))])
assert s_sim.shape == (11, 2, 2)
assert np.all(np.isfinite(s_sim))

comparison = {}
fig, axes = plt.subplots(4, 2, figsize=(12, 12), sharex=True)
for label, color in (("die_x2y7", "C0"), ("die_x5y8", "C1")):
    network = rf.Network(str(b.MEASURED[label]))
    assert np.allclose(network.z0, b.Z0_OHM)
    f_meas = network.f / 1e9
    assert f_meas[0] <= f_sim[0] and f_meas[-1] >= f_sim[-1] - 1e-9
    s_ref = np.empty_like(s_sim)
    for i, j, name in s_labels:
        trace = network.s[:, i, j]
        s_ref[:, i, j] = np.interp(f_sim, f_meas, trace.real) + 1j * np.interp(
            f_sim, f_meas, trace.imag
        )
        delta = s_sim[:, i, j] - s_ref[:, i, j]
        mag_db = lambda s: 20 * np.log10(np.maximum(np.abs(s), 1e-6))
        # Circular phase difference; omit phase where the reference is near a null.
        phase_mask = np.abs(s_ref[:, i, j]) >= 0.03
        phase_error = np.angle(
            s_sim[phase_mask, i, j] / s_ref[phase_mask, i, j], deg=True
        )
        comparison.setdefault(label, {})[name] = {
            "complex_rms": float(np.sqrt(np.mean(np.abs(delta) ** 2))),
            "magnitude_rms_db": float(
                np.sqrt(np.mean((mag_db(s_sim[:, i, j]) - mag_db(s_ref[:, i, j])) ** 2))
            ),
            "phase_rms_deg": float(np.sqrt(np.mean(phase_error**2)))
            if phase_error.size
            else None,
            "phase_points": int(phase_error.size),
        }
        row = [item[2] for item in s_labels].index(name)
        axes[row, 0].plot(
            f_meas, mag_db(trace), color=color, lw=1, label=f"measured {label}"
        )
        axes[row, 1].plot(
            f_meas,
            np.angle(trace, deg=True),
            color=color,
            lw=1,
            label=f"measured {label}",
        )
for row, (i, j, name) in enumerate(s_labels):
    trace = s_sim[:, i, j]
    axes[row, 0].plot(
        f_sim,
        20 * np.log10(np.maximum(np.abs(trace), 1e-6)),
        "o-",
        color="C3",
        ms=3,
        label="Palace",
    )
    axes[row, 1].plot(
        f_sim, np.angle(trace, deg=True), "o-", color="C3", ms=3, label="Palace"
    )
    axes[row, 0].set(ylabel=f"{name} (dB)")
    axes[row, 1].set(ylabel=f"{name} phase (deg)")
    for ax in axes[row]:
        ax.grid(alpha=0.3)
        ax.xaxis.set_major_locator(MaxNLocator(6))
axes[0, 0].legend(fontsize=7)
axes[0, 1].legend(fontsize=7)
axes[-1, 0].set(xlabel="frequency (GHz)")
axes[-1, 1].set(xlabel="frequency (GHz)")
fig.suptitle("Complete two-port comparison: measured .s2p vs Palace")
fig.tight_layout()
plt.show()

print("RMS errors at 11 Palace frequencies (1–50 GHz); phase excludes |Smeas| < 0.03")
for label, rows in comparison.items():
    print(label)
    for name, metrics in rows.items():
        print(
            f"  {name}: complex={metrics['complex_rms']:.3f}, magnitude={metrics['magnitude_rms_db']:.2f} dB, phase={metrics['phase_rms_deg']:.1f} deg ({metrics['phase_points']} points)"
        )

In [ ]:
# Diagnostic only: the pure-series S21 extraction used in the older campaign.
def lq_from_s(s, f_hz, z0=b.Z0_OHM):
    z = 2.0 * z0 * (1.0 - s[1, 0]) / s[1, 0]
    l_nh = z.imag / (2 * np.pi * f_hz) * 1e9
    q = z.imag / z.real if z.real != 0 else float("nan")
    return l_nh, q, z


def palace_curves():
    rows = []
    for i, f_ghz in enumerate(np.asarray(sp.freq, dtype=float)):
        s = b.s_matrix(sp, i)
        l, q, z = b.lq_from_s_matrix(s, f_ghz * 1e9)
        rows.append((f_ghz, l, q, s))
    return rows


def measured_curves(label):
    n = rf.Network(str(b.MEASURED[label]))
    z0 = float(np.real(np.asarray(n.z0).reshape(-1)[0]))
    z = 2.0 * z0 * (1.0 - n.s[:, 1, 0]) / n.s[:, 1, 0]
    l = z.imag / (2 * np.pi * n.f) * 1e9
    q = np.divide(z.imag, z.real, out=np.full_like(z.real, np.nan), where=z.real != 0)
    return n.f / 1e9, l, q, n.s


palace = palace_curves()
measured = {label: measured_curves(label) for label in ("die_x2y7", "die_x5y8")}

In [ ]:
# Primary IHP extraction: differential impedance of the complete two-port.
def differential_curve(f_hz, matrices):
    rows = [
        b.lq_from_impedance(b.differential_impedance_from_s(s), f)
        for f, s in zip(f_hz, matrices, strict=True)
    ]
    return np.asarray(rows, float).T


f_palace = np.asarray(sp.freq, float)
palace_l, palace_q = differential_curve(f_palace * 1e9, s_sim)
differential_measured = {}
for label in ("die_x2y7", "die_x5y8"):
    network = rf.Network(str(b.MEASURED[label]))
    assert np.allclose(network.z0, b.Z0_OHM)
    l, q = differential_curve(network.f, network.s)
    differential_measured[label] = (network.f / 1e9, l, q)

print(f"{'data':14} {'L at 1 GHz (nH)':>17} {'Q at 1 GHz':>13}")
print(f"{'Palace':14} {palace_l[0]:17.3f} {palace_q[0]:13.3f}")
for label, (f, l, q) in differential_measured.items():
    i = int(np.argmin(np.abs(f - 1.0)))
    print(f"{label:14} {l[i]:17.3f} {q[i]:13.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, color in (("die_x2y7", "C0"), ("die_x5y8", "C1")):
    f, l, q = differential_measured[label]
    axes[0].plot(f, l, color=color, lw=1, label=f"measured {label}")
    axes[1].plot(f, q, color=color, lw=1, label=f"measured {label}")
axes[0].plot(f_palace, palace_l, "o-", color="C3", label="Palace")
axes[1].plot(f_palace, palace_q, "o-", color="C3", label="Palace")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.set(xlabel="frequency (GHz)")
    ax.legend(fontsize=8)
axes[0].set(title="Differential impedance inductance", ylabel="L (nH)")
axes[1].set(title="Differential impedance quality factor", ylabel="Q")
fig.tight_layout()
plt.show()

In [ ]:
# Historical pure-series S21 diagnostic at 1 GHz only; not a device L/Q result.
import json

matrix = json.loads(
    (
        b.REPO / "nbs" / "data" / "ihp_validation" / "experiments" / "results.json"
    ).read_text()
)
for name in ("baseline", "best"):
    row = matrix["ground0_pec" if name == "best" else name]
    print(f"{name}: S21-series L={row['L_nH']:.3f} nH, Q={row['Q']:.2f}")

In [ ]:
# Historical S21-only pure-series diagnostic (not the primary L/Q comparison).
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, color in (("die_x2y7", "C0"), ("die_x5y8", "C1")):
    f, l, q, s = measured[label]
    axes[0].plot(f, l, color=color, lw=1.0, label=f"measured {label}")
    axes[1].plot(f, q, color=color, lw=1.0, label=f"measured {label}")
f_p = np.array([row[0] for row in palace])
axes[0].plot(f_p, [row[1] for row in palace], "o-", color="C3", label="Palace")
axes[1].plot(f_p, [row[2] for row in palace], "o-", color="C3", label="Palace")
for ax in axes:
    ax.axvline(30, color="k", lw=0.8, ls=":")
    ax.grid(alpha=0.3)
    ax.set(xlabel="frequency (GHz)")
    ax.legend(fontsize=8)
axes[0].set(title="S21-series diagnostic inductance", ylabel="L (nH)")
axes[1].set(title="S21-series diagnostic Q", ylabel="Q")
fig.tight_layout()
plt.show()

In [ ]:
# Diagnostic magnitude-only view; the complete complex comparison is above.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for label, color in (("die_x2y7", "C0"), ("die_x5y8", "C1")):
    f, l, q, s = measured[label]
    axes[0].plot(
        f, 20 * np.log10(np.abs(s[:, 0, 0])), color=color, lw=1.0, label=f"{label} S11"
    )
    axes[0].plot(
        f,
        20 * np.log10(np.abs(s[:, 1, 0])),
        color=color,
        lw=1.0,
        ls="--",
        label=f"{label} S21",
    )
    axes[1].plot(
        f, 20 * np.log10(np.abs(s[:, 0, 0])), color=color, lw=1.0, label=f"{label} S11"
    )
    axes[1].plot(
        f,
        20 * np.log10(np.abs(s[:, 1, 0])),
        color=color,
        lw=1.0,
        ls="--",
        label=f"{label} S21",
    )
for i, f_ghz in enumerate(np.asarray(sp.freq, dtype=float)):
    s = b.s_matrix(sp, i)
    axes[0].plot([f_ghz], [20 * np.log10(abs(s[0, 0]))], "o", color="C3")
    axes[0].plot([f_ghz], [20 * np.log10(abs(s[1, 0]))], "s", color="C3")
axes[0].set(
    title="Measured and Palace S-parameters", xlabel="frequency (GHz)", ylabel="dB"
)
axes[0].legend(fontsize=7, ncol=2)
axes[0].grid(alpha=0.3)
axes[1].set(
    title="Measured S-parameters (full band)", xlabel="frequency (GHz)", ylabel="dB"
)
axes[1].legend(fontsize=7, ncol=2)
axes[1].grid(alpha=0.3)
fig.tight_layout()
plt.show()

### Experiment matrix

The variants below are exploratory model and port sensitivity runs, not
validated device models. `ground0_pec` was selected after looking at the
measured 1 GHz series-extraction value. Its full S-matrix comparison above
shows the actual disagreement. The `pass` column means only that the selected
extracted inductance was finite and positive.


In [ ]:
import json
from pathlib import Path

matrix = json.loads(
    (
        b.REPO / "nbs" / "data" / "ihp_validation" / "experiments" / "results.json"
    ).read_text()
)
print(f"{'variant':18} {'L 1GHz':>8} {'Q':>7} {'L diff':>8} {'pass':>5}")
for name, row in matrix.items():
    print(
        f"{name:18} {row.get('L_nH', float('nan')):8.3f} {row.get('Q', float('nan')):7.2f} {row.get('L_diff_nH', float('nan')):8.3f} {str(row.get('pass')):>5}"
    )

### Findings and limitations

- The complete S-matrix disagrees with both measured dies in magnitude and
  phase. The table above quantifies each of S11, S21, S12 and S22 at all 11
  Palace frequencies. The model is not validated.
- The 1 GHz `ground0_pec` series-extraction value of 1.921 nH is close to the
  measured 1.911/1.932 nH, but the IHP differential extraction gives about
  0.80 nH for Palace versus about 1.67 nH measured. Low-frequency Q is about
  0.087 by differential extraction versus about 6.6 measured. The old
  series-only extraction gives Q = 0.21; its L match is misleading.
- The outer wall is unconditioned when the absorbing boundary is disabled; it
  should not be called PEC. Port geometry and ground return differ from the
  pinned OpenEMS model. Mesh error Norm is about 0.18 against AdaptiveTol 0.02.
- The 30 GHz point lies near/past a measured resonance, so a single extracted
  inductance there is not a meaningful validation metric. Compare the complex
  response above instead.


### Additional pinned IHP measurements and OpenEMS results

The IHP testcase also publishes **two measured differential one-port `.s1p`**
files (one per die), a two-port OpenEMS `.s2p` reference, and six one-port
OpenEMS `.s1p` outputs from Python/Matlab scripts at several mesh settings.
The one-port files use a single x-directed differential gap; they are shown
as their own experiment, not treated as extra two-port ports or as a
de-embedded equivalent of the Palace network. All files come from the pinned
IHP commit and are listed with SHA-256 hashes in
[`source/provenance.json`](data/ihp_validation/source/provenance.json).

The two-port OpenEMS reference uses the published edge-port/ground-pillar
geometry. It can be compared with the measured two-port response on its
0.1–30 GHz range, but the port/reference-plane match has not been
independently verified. The one-port Python and Matlab variants may differ
in geometry and stack details; their plot is a source inventory, not a
combined convergence curve.


In [ ]:
SOURCE = b.SOURCE
ihp_twoport = rf.Network(str(SOURCE / "openems_twoport_1p5um.s2p"))
assert ihp_twoport.nports == 2 and np.allclose(ihp_twoport.z0, 50)

fig, axes = plt.subplots(4, 2, figsize=(12, 12), sharex=True)
for row, (i, j, name) in enumerate(s_labels):
    for die, color in (("die_x2y7", "C0"), ("die_x5y8", "C1")):
        net = rf.Network(str(b.MEASURED[die]))
        axes[row, 0].plot(
            net.f / 1e9,
            20 * np.log10(np.maximum(abs(net.s[:, i, j]), 1e-6)),
            color=color,
            label=die,
        )
        axes[row, 1].plot(
            net.f / 1e9, np.angle(net.s[:, i, j], deg=True), color=color, label=die
        )
    trace = ihp_twoport.s[:, i, j]
    axes[row, 0].plot(
        ihp_twoport.f / 1e9,
        20 * np.log10(np.maximum(abs(trace), 1e-6)),
        "k--",
        lw=1,
        label="IHP OpenEMS 1.5 µm",
    )
    axes[row, 1].plot(
        ihp_twoport.f / 1e9,
        np.angle(trace, deg=True),
        "k--",
        lw=1,
        label="IHP OpenEMS 1.5 µm",
    )
    axes[row, 0].set(ylabel=f"{name} (dB)", xlim=(0, 30))
    axes[row, 1].set(ylabel=f"{name} phase (deg)", xlim=(0, 30))
    for ax in axes[row]:
        ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=7)
axes[-1, 0].set(xlabel="frequency (GHz)")
axes[-1, 1].set(xlabel="frequency (GHz)")
fig.suptitle("L2n0 measured two-port and pinned IHP OpenEMS result")
fig.tight_layout()
plt.show()

oneport_names = {
    "measured x2y7": "diff_die_x2y7.s1p",
    "measured x5y8": "diff_die_x5y8.s1p",
    "Python 1.5 µm": "openems_oneport_1p5um.s1p",
    "Python 1 µm": "openems_oneport_1um.s1p",
    "Matlab 1.5 µm": "openems_matlab_oneport_1p5um.s1p",
    "Matlab 1 µm": "openems_matlab_oneport_1um.s1p",
    "Matlab 0.5 µm": "openems_matlab_oneport_0p5um.s1p",
    "Matlab 0.5 µm finer z": "openems_matlab_oneport_0p5um_finer_z.s1p",
}
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for label, filename in oneport_names.items():
    net = rf.Network(str(SOURCE / filename))
    assert net.nports == 1 and np.allclose(net.z0, 50)
    row = 0 if label.startswith(("measured", "Python")) else 1
    color = "k" if label.startswith("measured") else None
    style = "-" if label.startswith("measured") else "--"
    axes[row, 0].plot(
        net.f / 1e9,
        20 * np.log10(np.maximum(abs(net.s[:, 0, 0]), 1e-6)),
        style,
        color=color,
        lw=1,
        label=label,
    )
    axes[row, 1].plot(
        net.f / 1e9,
        np.angle(net.s[:, 0, 0], deg=True),
        style,
        color=color,
        lw=1,
        label=label,
    )
# Repeat both measured dies in the Matlab row for a direct visual comparison.
for die, color in (("x2y7", "k"), ("x5y8", "0.45")):
    net = rf.Network(str(SOURCE / f"diff_die_{die}.s1p"))
    axes[1, 0].plot(
        net.f / 1e9,
        20 * np.log10(np.maximum(abs(net.s[:, 0, 0]), 1e-6)),
        color=color,
        lw=1,
        label=f"measured {die}",
    )
    axes[1, 1].plot(
        net.f / 1e9,
        np.angle(net.s[:, 0, 0], deg=True),
        color=color,
        lw=1,
        label=f"measured {die}",
    )
for row in (0, 1):
    for ax in axes[row]:
        ax.set(xlim=(0, 30))
        ax.grid(alpha=0.3)
    axes[row, 0].set(ylabel="S11 (dB)")
    axes[row, 1].set(ylabel="S11 phase (deg)")
    axes[row, 0].legend(fontsize=7)
axes[0, 0].set_title("Measured dies and Python OpenEMS")
axes[1, 0].set_title("Matlab OpenEMS mesh variants")
axes[1, 0].set_xlabel("frequency (GHz)")
axes[1, 1].set_xlabel("frequency (GHz)")
fig.tight_layout()
plt.show()

The two measured `.s2p` files and two differential `.s1p` files are distinct
measurement products for the same nominal L2n0 device, not four independent
dies. One-port S11 cannot be substituted for a single entry of the two-port
S-matrix.
